# NVTTS evaluator trên Kaggle

**Trước khi chạy** (panel bên phải):
1. **Internet: On** (cần để `pip`, tải model từ Hugging Face và `git`).
2. **Add Input**: dataset `vinv-tts-train` (có `vinv-tts/train` và `vinv-tts/dev`).
3. **Accelerator**: GPU T4 là tùy chọn (tăng tốc DNSMOS và ECAPA; ASR luôn chạy trên CPU).

Chạy từ trên xuống. Mọi ô đều chạy lại an toàn: ô nào đã có kết quả sẽ bỏ qua hoặc dùng cache.

### 0. Cài đặt (clone hoặc cập nhật repo)

In [ ]:
REPO = "/kaggle/working/NVTTS-Evaluator"
# repo đã có thì pull, chưa có thì clone
!test -d {REPO}/.git && git -C {REPO} pull || git clone --depth 1 https://github.com/VietCH57/NVTTS-Evaluator.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q -e ".[dev,asr,mos,asv,nvpa]"
!python -m pytest -q          # kỳ vọng: 147 passed

### 1. Mô hình ASR (chỉ tải lần đầu)

In [ ]:
import subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

ASR_DIR = "/kaggle/working/models/Zipformer-30M-RNNT-6000h"
if not (Path(ASR_DIR) / "tokens.txt").exists():
    snapshot_download("hynt/Zipformer-30M-RNNT-6000h",
        allow_patterns=["encoder-epoch-20-avg-10.onnx", "decoder-epoch-20-avg-10.onnx",
                        "joiner-epoch-20-avg-10.onnx", "bpe.model"],
        local_dir=ASR_DIR)
    subprocess.run(["python", "scripts/make_tokens.py", "--bpe", f"{ASR_DIR}/bpe.model",
                    "--out", f"{ASR_DIR}/tokens.txt"], check=True)
print(sorted(p.name for p in Path(ASR_DIR).glob("*")))     # phải có tokens.txt (2000 token)

### 2. Dữ liệu và manifest

In [ ]:
DATA = "/kaggle/input/datasets/hongvitchugo/vinv-tts-train/vinv-tts"
!ls {DATA}
!ls {DATA}/train | wc -l      # 189 speaker
!ls {DATA}/dev | wc -l        # 46 speaker

# dev: dùng để đánh giá (có reference cho SS). train: chỉ để huấn luyện detector NVPA (không cần reference)
!python -m nvtts_eval.data.adapter_vinv --root {DATA} --split dev --track A --out /kaggle/working/manifests/dev_gt_A.jsonl
!python -m nvtts_eval.data.adapter_vinv --root {DATA} --split train --track A --no-references --out /kaggle/working/manifests/train_gt.jsonl
# kỳ vọng: samples=316 (dev) và samples=1739 (train), missing_ground_truth=0

### 3. Cấu hình

In [ ]:
BASE_CFG = """\
asr:
  model_dir: /kaggle/working/models/Zipformer-30M-RNNT-6000h
  num_threads: 4
pmos:
  device: auto
ss:
  savedir: /kaggle/working/models/spkrec-ecapa-voxceleb
  device: auto
  max_reference_clips: 20
"""
# Detector NVPA: huấn luyện ở bước 5. Nếu dùng bản đã lưu trong một Kaggle Dataset, đổi thành
# "/kaggle/input/<ten-dataset>/nvpa_gap.joblib"
DETECTOR = "/kaggle/working/models/nvpa_gap.joblib"

# Điểm vận hành NVPA chọn sau khi đọc nvpa-sweep (bước 5d/5e). Để trống = dung sai ±1 từ, ngưỡng gốc.
NVPA_EXTRA = ""

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

def write_configs():
    Path("/kaggle/working/kaggle.yaml").write_text(BASE_CFG, encoding="utf-8")                   # chưa có NVPA
    Path("/kaggle/working/kaggle_nvpa.yaml").write_text(
        BASE_CFG + f"nvpa:\n  detector_path: {DETECTOR}\n" + NVPA_EXTRA, encoding="utf-8")   # có NVPA

write_configs()
print(Path("/kaggle/working/kaggle_nvpa.yaml").read_text())

### 4. Kiểm tra ba model trên dữ liệu thật (tùy chọn, chạy lần đầu)

In [ ]:
!python scripts/probe_models.py --manifest /kaggle/working/manifests/dev_gt_A.jsonl --config /kaggle/working/kaggle.yaml --n 3

### 5. Calibration: audio thật của dev đóng vai "output của model"

**5a. ASR + pMOS + SS** (khoảng 6 phút; lần sau dùng cache).

In [ ]:
!python -m nvtts_eval.cli run --manifest /kaggle/working/manifests/dev_gt_A.jsonl --run-dir /kaggle/working/runs/gt_dev_A --config /kaggle/working/kaggle.yaml

**5b. Huấn luyện detector NVPA** trên tập **train** (ASR chạy qua 5.9 giờ audio, khoảng 15 phút, rồi vài phút huấn luyện).
Lệnh từ chối chạy trên dev. Cuối ô sẽ in precision/recall theo từng loại NV: hãy đọc kỹ.

In [ ]:
!python -m nvtts_eval.cli train-detector --manifest /kaggle/working/manifests/train_gt.jsonl --run-dir /kaggle/working/runs/train_gt --config /kaggle/working/kaggle.yaml --out {DETECTOR}

**5c. NVPA trên dev** (ASR, pMOS, SS lấy từ cache; chỉ NVPA phải tính) và báo cáo.

In [ ]:
!python -m nvtts_eval.cli run --manifest /kaggle/working/manifests/dev_gt_A.jsonl --run-dir /kaggle/working/runs/gt_dev_A --config /kaggle/working/kaggle_nvpa.yaml --metrics nvpa

In [ ]:
import json
s = json.load(open("/kaggle/working/runs/gt_dev_A/summary.json", encoding="utf-8"))
n = s["automatic_metrics"]["nvpa"]
print("NVPA trên audio thật:", round(n["value"], 3), "| mức chọn vị trí ngẫu nhiên:", round(n["shuffle_baseline"]["mean"], 3))
print("theo loại:", {t: (round(v["mean"], 3), v["n"]) for t, v in n["by_type"].items()})
print("lý do trượt:", n["reasons"], "| NV thừa/100 từ:", round(n["spurious_per_100_words"], 2))
print("AutoScore (trần của audio thật):", s["automatic_score"] and round(s["automatic_score"]["value"], 4), "/ tối đa 0.70")

**5d. Chẩn đoán NVPA (không chạy lại model).** Quét dung sai vị trí và ngưỡng detector từ artifact đã lưu. Đọc cột **lift**
(NVPA trừ mức ngẫu nhiên), không đọc riêng NVPA: nới dung sai hay hạ ngưỡng đều làm tăng cả hai.
Dung sai viết `n` (đối xứng ±n từ) hoặc `trước:sau` (ví dụ `0:1` = phát hiện được phép MUỘN 1 từ nhưng không được sớm).
Dòng cuối là histogram độ lệch có dấu (vị trí phát hiện trừ vị trí gold, tính bằng từ): nếu lệch về một phía thì có sai số hệ thống.

In [ ]:
!python -m nvtts_eval.cli nvpa-sweep --run-dir /kaggle/working/runs/gt_dev_A --config /kaggle/working/kaggle_nvpa.yaml --tolerances 0 1 0:1 1:0 2 --scales 0.5 0.75 1.0

**5e. Áp dụng điểm vận hành đã chọn.** Đọc bảng 5d rồi chọn dòng có **lift** cao và NV thừa/100 từ chấp nhận được
(mật độ NV gold là 5.4 trên 100 từ). Giá trị gợi ý ban đầu dưới đây dựa trên lần sweep trước (độ lệch +1 gấp 55 lần −1;
ngưỡng ×0.75 cho lift cao nhất ở dung sai 1): hãy kiểm tra lại bằng bảng mới trước khi bật. Đổi cấu hình sẽ tự làm cache NVPA hết hiệu lực.

In [ ]:
APPLY_CALIBRATION = False        # đổi thành True sau khi đã chọn từ bảng sweep
NVPA_EXTRA = "  tolerance_before: 0\n  tolerance_after: 1\n  threshold_scale: 0.75\n"
if APPLY_CALIBRATION:
    write_configs()
    sh("python -m nvtts_eval.cli run --manifest /kaggle/working/manifests/dev_gt_A.jsonl --run-dir /kaggle/working/runs/gt_dev_A --config /kaggle/working/kaggle_nvpa.yaml --metrics nvpa")

### 6. Lưu kết quả để dùng lại
Session Kaggle tắt là mất `/kaggle/working` (trừ khi **Save Version**). Hãy tải về (hoặc tạo một **Dataset private** từ) hai file dưới đây,
đặc biệt `nvpa_gap.joblib`, để lần sau khỏi huấn luyện lại (đổi `DETECTOR` ở bước 3 sang đường dẫn trong `/kaggle/input/...`).

In [ ]:
import shutil
Path("/kaggle/working/export").mkdir(exist_ok=True)
shutil.copy(DETECTOR, "/kaggle/working/export/nvpa_gap.joblib")
shutil.copy("/kaggle/working/runs/gt_dev_A/summary.json", "/kaggle/working/export/summary_gt_dev_A.json")
print(sorted(p.name for p in Path("/kaggle/working/export").iterdir()))

---
## (Tùy chọn) Đánh giá một model TTS
Đặt output của model vào một Kaggle Dataset, mỗi mẫu một file tên `<spk_id>_<tên file audio>.wav` (ví dụ `spk_0000_0001.wav`).
Track B cần thêm `--track B --reference-map refs.json` (xem README).

In [ ]:
RUN_MODEL_EVAL = False       # đổi thành True sau khi sửa GEN bên dưới
GEN = "/kaggle/input/<ten-dataset-output-cua-model>"      # SỬA
MODEL_M = "/kaggle/working/manifests/mymodel_A.jsonl"
MODEL_RUN = "/kaggle/working/runs/mymodel_A"

if RUN_MODEL_EVAL:
    sh(f"python -m nvtts_eval.data.adapter_vinv --root {DATA} --split dev --track A --source model --generated-dir {GEN} --out {MODEL_M}")
    sh(f"python -m nvtts_eval.cli run --manifest {MODEL_M} --run-dir {MODEL_RUN} --config /kaggle/working/kaggle_nvpa.yaml")

## (Tùy chọn) Đánh giá bởi người
Chỉ gửi cho người chấm thư mục `for_raters`; **giữ kín `PRIVATE_key.json`** (tải về máy ngay sau khi xuất, vì cần nó khi nhập điểm).
Mỗi người chấm điền một bản `rating_sheet.csv` và gửi lại với tên theo người chấm (`alice.csv`, `bob.csv`).
Khi các rater có điền cột `NV_placement`, báo cáo còn in tương quan Spearman giữa NVPA và điểm của người: đây là phép kiểm chứng
detector NVPA đáng tin nhất.

In [ ]:
RUN_HUMAN_EXPORT = False     # đổi thành True sau khi đã có manifest của model (ô trên)
CFG = "/kaggle/working/kaggle_nvpa.yaml"
PKG = "/kaggle/working/human_package"
if RUN_HUMAN_EXPORT:
    import shutil
    sh(f"python -m nvtts_eval.cli human-subset --manifest {MODEL_M} --run-dir {MODEL_RUN} --config {CFG} --size 100")
    sh(f"python -m nvtts_eval.cli human-export --manifest {MODEL_M} --run-dir {MODEL_RUN} --config {CFG} --out {PKG}")
    shutil.make_archive("/kaggle/working/for_raters", "zip", PKG, "for_raters")       # file zip gửi cho người chấm
    print("Tải về: /kaggle/working/for_raters.zip và", PKG + "/PRIVATE_key.json")

In [ ]:
# Sau khi có điểm: đưa alice.csv, bob.csv ... vào /kaggle/working/ratings/ (hoặc một Kaggle Dataset)
RUN_HUMAN_IMPORT = False
RATINGS = "/kaggle/working/ratings"
if RUN_HUMAN_IMPORT:
    sh(f"python -m nvtts_eval.cli human-import --manifest {MODEL_M} --run-dir {MODEL_RUN} --config {CFG} --package {PKG} --ratings {RATINGS}/alice.csv {RATINGS}/bob.csv")